## Incident Clustering (Agrupamento inteligente de incidentes)

Este notebook identifica grupos de incidentes semelhantes sem utilizar rótulos previamente definidos. O clustering utiliza exclusivamente a coluna **Descrição resumida** para descobrir padrões recorrentes. O número do incidente é mantido somente para identificação no resultado final.

O tratamento textual inclui conversão para minúsculas, remoção de caracteres especiais, remoção de identificadores técnicos como `IC00001` e `INC1234567`, remoção de stopwords em português e inglês, tratamento de nulos e stemming em inglês quando disponível.

São comparadas representações TF-IDF por palavras com n-gramas e por caracteres, combinadas com diferentes valores de `k` no MiniBatchKMeans. A configuração final é escolhida por um score combinado de rankings de Silhouette, Davies-Bouldin e Calinski-Harabasz.


In [1]:
from pathlib import Path
import re
import json
import unicodedata
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

from sklearn.metrics import (silhouette_score, davies_bouldin_score, calinski_harabasz_score)

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

try:
    from nltk.stem.snowball import SnowballStemmer
    STEMMER = SnowballStemmer("english")
except:
    STEMMER = None

In [2]:
INPUT = "LW-DATASET-TRATADO.xlsx"

OUT = Path("outputs/incident_clustering")
OUT.mkdir(parents=True, exist_ok=True)

### Remoção de Stopwords

Para melhorar a qualidade da clusterização, foi criada uma lista personalizada de stopwords (palavras com baixa capacidade de diferenciação entre incidentes)

Essa lista combina:
- Stopwords em português
- Stopwords em inglês
- Termos frequentes no contexto de monitoramento
- Palavras genéricas presentes em praticamente todos os incidentes

Sem essa remoção, o modelo poderia agrupar incidentes por palavras muito comuns, reduzindo sua capacidade de identificar padrões operacionais relevantes.

In [3]:
STOP = set("""
a an and as at be by com da das de do dos e em for from is na nas
no nos not o os ou para por que se sem the to um uma uns umas
was were with this that on of in are no yes alarm problem check
application monitoring error message host port time timeout high
low free space lack processor load backup disk unavailable http https
type running grown up cpu queue
""".split())

### Tratamento e Limpeza do Texto

Antes da aplicação das técnicas de NLP, foi realizado um processo de limpeza das descrições dos incidentes para reduzir ruídos e padronizar o texto.

- Padroniza a escrita e evita que palavras como: disk, Disk e DISK sejam tratadas como termos diferentes.
- Remove identificadores que não agregam significado semântico para a clusterização.
- Padroniza palavras acentuadas (indisponível --> indisponivel)
- Mantém apenas letras e espaços, removendo caracteres especiais
- Tokenização e filtragem: texto é dividido em palavras individuais para processamento
- Remoção de palavras muito curtas (até duas letras geralmente possuem pouco valor semântico)
- Remoção de stopwords (palavras extremamente frequentes que não ajudam a diferenciar)
- Stemming: Reduz palavras para sua raiz linguística (running --> run; workers --> worker)
- Remoção de URLs e e-mails

In [4]:
def clean(text):
    text = "" if pd.isna(text) else str(text)
    text = text.lower()

    text = re.sub(r"\b(?:inc|ic|team|vm|srv|host|id)[\s_-]*\d+[a-z0-9_-]*\b", " ", text)
    text = re.sub(r"https?://\S+|\S+@\S+", " ", text)

    text = (unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii"))

    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)

    tokens = []

    for word in text.split():
        if len(word) <= 2:
            continue

        if word in STOP:
            continue

        if STEMMER:
            word = STEMMER.stem(word)

        tokens.append(word)

    return " ".join(tokens)

Leitura das informações necessárias para clusterização: nº incidente e descrição (nomes foram padronizados)

- descrições nulas foram substituídas por texto vazio
- foi aplicada a função de tratamento criada anteriormente

In [5]:
df = pd.read_excel(INPUT, usecols=["Número", "Descrição resumida"])
df = df.rename(columns={"Número": "numero_incidente", "Descrição resumida": "descricao_original"})

df["descricao_original"] = (df["descricao_original"].fillna("").astype(str))

df["texto_limpo"] = df["descricao_original"].apply(clean)

df = df[df["texto_limpo"].str.len() > 0].reset_index(drop=True)
df.shape

(93687, 3)

### Criação do Score Geral para Seleção do Melhor Modelo

Como diferentes métricas de clusterização possuem escalas distintas, foi criada uma função responsável por transformar cada métrica em um ranking normalizado.

As métricas avaliadas possuem comportamentos e escalas diferentes:
| Métrica | Melhor Resultado | Escala |
|----------|----------|-------|
| Silhouette Score | Maior |  0.2 a 0.5 |
| Davies-Bouldin Index | Menor | 0.9 a 2 |
| Calinski-Harabasz Score | Maior | centenas ou milhares |

Comparar diretamente poderia gerar uma escolha errada.

#### A função converte os resultados em um ranking
- Quando higher=true: os maiores valores recebem melhores posições
- Quando higher=false: os menores valores recebem melhores posições 

Dessa forma, todas as métricas passam a possuir uma escala semelhante, variando aproximadamente entre:
- Melhor resultado -> próximo de 0
- Pior resultado -> próximo de 1

O modelo escolhido é aquele com o menor valor de `score_geral`.

In [6]:
def rank01(series, higher=True):
    rank = series.rank(method="min", ascending=not higher)
    return rank / rank.max()

## Construção dos Clusters

Após o pré-processamento dos textos, foram testadas diferentes formas de representar as descrições dos incidentes para identificar a combinação com melhor qualidade de agrupamento.

### Estratégias avaliadas

**TF-IDF (1-2 gramas)**
- Palavras isoladas e combinações de duas palavras.

**TF-IDF (1-3 gramas)**
- Inclui também sequências de três palavras para capturar mais contexto.

**TF-IDF por caracteres**
- Utiliza sequências de 3 a 5 caracteres.
- Ajuda a capturar abreviações, variações de escrita e termos semelhantes.

### Quantidade de clusters

Foram avaliados diferentes valores de K para encontrar grupos mais coerentes e representativos dos incidentes.

### Algoritmo

Foi utilizado o **MiniBatch K-Means**, escolhido por sua eficiência em bases com grande volume de registros.

### Avaliação

Para cálculo das métricas, foi aplicada redução de dimensionalidade com **Truncated SVD**. A clusterização foi realizada utilizando a matriz TF-IDF completa.

### Métricas

- **Silhouette Score:** quanto maior, melhor.
- **Davies-Bouldin Index:** quanto menor, melhor.
- **Calinski-Harabasz Score:** quanto maior, melhor.

In [7]:
configs = []

vectorizers = [
    (
        "word_1_2",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            max_features=30000)),
    (
        "word_1_3",
        TfidfVectorizer(
            ngram_range=(1, 3),
            min_df=2,
            max_df=0.98,
            sublinear_tf=True,
            max_features=30000)),
    (
        "char_3_5",
        TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=3,
            sublinear_tf=True,
            max_features=30000)),]

for nome, vec in vectorizers:
    X = vec.fit_transform(df["texto_limpo"])

    for k in [4, 8, 10, 12]:
        model = MiniBatchKMeans(
            n_clusters=k,
            random_state=42,
            n_init=10,
            batch_size=2048)

        labels = model.fit_predict(X)

        sample_idx = (
            np.arange(len(df))
            if len(df) <= 10000
            else np.random.default_rng(42).choice(len(df), 10000, replace=False))

        X_sample = X[sample_idx]
        labels_sample = labels[sample_idx]

        svd = TruncatedSVD(
            n_components=100,
            random_state=42)

        X_eval = svd.fit_transform(X_sample)

        configs.append(
            {"config": nome,
                "k": k,
                "silhouette": silhouette_score(X_sample, labels_sample, metric="cosine"),
                "davies_bouldin": davies_bouldin_score(X_eval, labels_sample),
                "calinski_harabasz": calinski_harabasz_score(X_eval, labels_sample),
                "vectorizer": vec, "X": X,
                "labels": labels, "model": model })

## Comparação das Configurações Testadas

Lembrando que:

| Métrica | Melhor Resultado |
|----------|----------|
| Silhouette Score | Maior |
| Davies-Bouldin Index | Menor |
| Calinski-Harabasz Score | Maior |

Para tornar as métricas comparáveis, foi aplicada a função `rank01()`, que converte cada métrica em um ranking normalizado.

Por fim, as configurações foram ordenadas pelo menor valor de `score_geral`.

Empate entre k=6 e k=8

In [8]:
res = pd.DataFrame([
    {k: v
        for k, v in c.items()
        if k not in ["vectorizer", "X", "labels", "model"]} 
        for c in configs])

res["rank_sil"] = rank01(res["silhouette"], True)
res["rank_db"] = rank01(res["davies_bouldin"], False)

res["rank_ch"] = rank01(res["calinski_harabasz"], True)
res["score_geral"] = (res[["rank_sil", "rank_db", "rank_ch"]].mean(axis=1))
res.sort_values("score_geral").head(10) 

,config,k,silhouette,davies_bouldin,calinski_harabasz,rank_sil,rank_db,rank_ch,score_geral
5,word_1_3,8,0.328064,1.082070,996.839459,0.583333,0.083333,0.166667,0.277778
0,word_1_2,4,0.207331,1.088630,1058.665139,0.833333,0.166667,0.083333,0.361111
11,char_3_5,12,0.424860,1.766574,945.835584,0.083333,0.750000,0.333333,0.388889
3,word_1_2,12,0.395496,1.286887,901.452137,0.250000,0.500000,0.500000,0.416667
10,char_3_5,10,0.395904,1.866913,976.479735,0.166667,0.833333,0.250000,0.416667
2,word_1_2,10,0.357894,1.155144,892.690317,0.416667,0.333333,0.583333,0.444444
6,word_1_3,10,0.351750,1.289460,903.001159,0.500000,0.583333,0.416667,0.500000
7,word_1_3,12,0.381093,1.704423,865.011878,0.333333,0.666667,0.750000,0.583333
4,word_1_3,4,0.172238,1.107536,828.100727,1.000000,0.250000,0.916667,0.722222
1,word_1_2,8,0.294048,1.175707,805.567048,0.750000,0.416667,1.000000,0.722222


Melhor Config

In [9]:
best_idx = res["score_geral"].idxmin()
best = configs[best_idx]

print(best["config"])
print(best["k"])

word_1_3
8


Aplicação de Melhor Config

In [10]:
labels = best["labels"]
vectorizer = best["vectorizer"]
X = best["X"]
model = best["model"]

df["cluster"] = labels

Keywords dos Clusters

In [11]:
terms = np.array(vectorizer.get_feature_names_out())

keywords_rows = []
summary_rows = []

for cluster in range(best["k"]):
    idx = np.where(labels == cluster)[0]
    centroid = model.cluster_centers_[cluster]
    top_idx = np.argsort(centroid)[::-1][:15]
    top_words = terms[top_idx]
    for palavra_idx in top_idx:

        keywords_rows.append({"cluster": cluster, "palavra": terms[palavra_idx], "score": centroid[palavra_idx] })

    summary_rows.append({
            "cluster": cluster,
            "quantidade_incidentes": len(idx),
            "percentual_dataset":
                len(idx) / len(df) * 100,

            "tema_cluster":
                " | ".join(top_words[:5])})

Descrição Representativa

In [12]:
X_norm = normalize(X)
C_norm = normalize(model.cluster_centers_)

representantes = {}

for cluster in range(best["k"]):
    idx = np.where(labels == cluster)[0]
    sims = X_norm[idx].dot(C_norm[cluster])
    melhor = idx[int(np.argmax(sims))]
    representantes[cluster] = (df.iloc[melhor]["descricao_original"])

Dataframes Finais

In [13]:
cluster_summary = pd.DataFrame(summary_rows)
cluster_summary["descricao_representativa"] = cluster_summary["cluster"].map(representantes)
cluster_keywords = pd.DataFrame(keywords_rows)
incident_clusters = df[["numero_incidente", "descricao_original", "cluster"]]

Exportação

In [14]:
incident_clusters.to_csv(OUT / "incident_clusters.csv", index=False, encoding="utf-8-sig")
cluster_summary.to_csv(OUT / "cluster_summary.csv", index=False, encoding="utf-8-sig")
cluster_keywords.to_csv(OUT / "cluster_keywords.csv", index=False, encoding="utf-8-sig")
res.to_csv(OUT / "clustering_model_comparison.csv", index=False, encoding="utf-8-sig")

print("Arquivos salvos!")

Arquivos salvos!


In [15]:
print("Melhor Configuração")

print("Vectorizer:", best["config"])
print("Clusters:", best["k"])

display(cluster_summary.sort_values("quantidade_incidentes", ascending=False).head(20))

Melhor Configuração
Vectorizer: word_1_3
Clusters: 8


,cluster,quantidade_incidentes,percentual_dataset,tema_cluster,descricao_representativa
0,0,60341,64.407015,vip | tcp | bacula | falha | down,Problem: Check Application Monitoring VIP
4,4,9104,9.717463,less than | than | less | than volume | less t...,Problem: Free disk space is less than 20% on v...
2,2,7457,7.959482,overloaded | able persist | able | abertura ch...,Problem: Disk I/O is overloaded on IC00006
7,7,4134,4.412565,icmp ping | ping | icmp | pleskw | abertura ch...,Problem: Unavailable by ICMP ping
6,6,3925,4.189482,busy workers | busy | apache busy workers | ap...,Problem: Apache Busy Workers
1,1,3663,3.909827,swap | swap memory server | available swap mem...,Problem: Lack of free swap space 40m <5%
3,3,2950,3.148783,too | too iowait | iowait | too many | many,Problem: Processor load is too high > 20%
5,5,2113,2.255382,hypervisor | erro | servidores erro hypervisor...,Problem: IC06745 - IC01442 - Servidores com er...


In [16]:
best_idx = res["score_geral"].idxmin()
best = configs[best_idx]
print("MELHOR CONFIGURAÇÃO ENCONTRADA")

print(f"Vectorizer: {best['config']}")
print(f"Número de Clusters: {best['k']}")

print("\nMÉTRICAS")
print(f"Silhouette Score: {res.loc[best_idx, 'silhouette']:.4f}")
print(f"Davies-Bouldin Index: {res.loc[best_idx, 'davies_bouldin']:.4f}")
print(f"Calinski-Harabasz Score: {res.loc[best_idx, 'calinski_harabasz']:.2f}")
print(f"Score Geral: {res.loc[best_idx, 'score_geral']:.6f}")

MELHOR CONFIGURAÇÃO ENCONTRADA
Vectorizer: word_1_3
Número de Clusters: 8

MÉTRICAS
Silhouette Score: 0.3281
Davies-Bouldin Index: 1.0821
Calinski-Harabasz Score: 996.84
Score Geral: 0.277778
